# Phase 1 – Data understanding
## Step 0: Load all tables

In [1]:
import pandas as pd
import os

DATA = r"C:\Users\start\OneDrive\Desktop\HVIA-TASK3\Cottonil_project\COTTONIL\COTTONIL_Data\02_TABLES"
pd.set_option("display.max_columns", 50)

In [2]:
tables = {}
for f in sorted(os.listdir(DATA)):
    if f.endswith(".csv"):
        tables[f.replace(".csv", "")] = pd.read_csv(os.path.join(DATA, f), low_memory=False)

overview = pd.DataFrame([
    {"table": n, "rows": len(df), "columns": df.shape[1]}
    for n, df in tables.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)
print("Total rows:", f"{overview.rows.sum():,}")
overview

Total rows: 2,931,843


,table,rows,columns
0,sales_order_items,1138536,13
1,sales_orders,540000,13
2,sales_payments,540000,6
3,monthly_stock,142784,7
4,inventory_transfer_items,109614,4
5,purchase_order_items,103334,7
6,delivery_shipments,97660,8
7,customers,85000,6
8,operating_expenses,47000,7
9,purchase_orders,41500,9


## Step 1: `branches`

In [3]:
b = tables["branches"]
print(b.shape)
b.head()

(80, 7)


,branch_id,branch_name_ar,governorate_ar,city_ar,address_ar,retail_location_type,opening_date
0,BR001,الزمالك,القاهرة,الزمالك,شارع 26 يوليو، الزمالك,Point of sale / reseller,9/14/2019
1,BR002,السيدة زينب,القاهرة,السيدة زينب,229 شارع بورسعيد، السيدة زينب,Point of sale / reseller,2/19/2019
2,BR003,القطامية داون تاون,القاهرة,القطامية,شارع التسعين، مول داون تاون، القطامية,Retail branch,2/12/2020
3,BR004,الرحاب مول,القاهرة,الرحاب,المرحلة الثالثة، مول الرحاب,Point of sale / reseller,8/14/2019
4,BR005,المعادي جراند مول,القاهرة,المعادي,شارع 250، جراند مول، المعادي,Retail branch,11/16/2018


In [4]:
print("Empty cells:", b.isna().sum().sum())
print("Duplicate branch_id:", b.branch_id.duplicated().sum())
print("Unique governorates:", b.governorate_ar.nunique(), "| Unique cities:", b.city_ar.nunique())

Empty cells: 0
Duplicate branch_id: 0
Unique governorates: 21 | Unique cities: 65


In [5]:
print(b.retail_location_type.value_counts(), "\n")
b.governorate_ar.value_counts().to_frame("branches")

retail_location_type
Retail branch               52
Point of sale / reseller    28
Name: count, dtype: int64 



,branches
governorate_ar,
القاهرة,21
الإسكندرية,10
الجيزة,6
الدقهلية,4
البحيرة,4
الغربية,4
المنيا,4
الشرقية,3
المنوفية,3


In [6]:
# branches inside malls
mall = b[b.branch_name_ar.str.contains("مول") | b.address_ar.str.contains("مول")]
print("Branches inside malls:", len(mall))
mall[["branch_id", "branch_name_ar", "governorate_ar", "retail_location_type"]]

Branches inside malls: 8


,branch_id,branch_name_ar,governorate_ar,retail_location_type
2,BR003,القطامية داون تاون,القاهرة,Retail branch
3,BR004,الرحاب مول,القاهرة,Point of sale / reseller
4,BR005,المعادي جراند مول,القاهرة,Retail branch
5,BR006,فيوتشر مول,القاهرة,Retail branch
18,BR019,كايرو فيستيفال,القاهرة,Retail branch
50,BR051,الأردنية,الشرقية,Retail branch
69,BR070,مول الصفوة,دمياط,Retail branch
79,BR080,الجامعة,أسيوط,Retail branch


In [7]:
b["opening_date"] = pd.to_datetime(b.opening_date, format="%m/%d/%Y")
first_sale = pd.to_datetime(tables["sales_orders"].groupby("branch_id").order_date.min())

check = b[["branch_id", "branch_name_ar", "opening_date"]].copy()
check["first_sale"] = check.branch_id.map(first_sale)
check["sold_before_opening_days"] = (check.opening_date - check.first_sale).dt.days

print("Opening dates range:", check.opening_date.min().date(), "->", check.opening_date.max().date())
print("First sale range:   ", check.first_sale.min().date(), "->", check.first_sale.max().date())
print("Branches that sold BEFORE their opening date:", (check.sold_before_opening_days > 0).sum(), "of", len(check))
check.sort_values("sold_before_opening_days", ascending=False).head()

Opening dates range: 2018-11-06 -> 2020-08-29
First sale range:    2018-10-19 -> 2018-10-22
Branches that sold BEFORE their opening date: 80 of 80


,branch_id,branch_name_ar,opening_date,first_sale,sold_before_opening_days
16,BR017,زهراء المعادي,2020-08-29,2018-10-19,680
40,BR041,طلخا,2020-08-26,2018-10-19,677
77,BR078,شرق البندر,2020-08-10,2018-10-19,661
11,BR012,البطراوي,2020-08-07,2018-10-20,657
55,BR056,سقالة,2020-07-26,2018-10-19,646


In [8]:
# orders that came BEFORE their branch opened, by sales channel
so = tables["sales_orders"]
x = so.merge(b[["branch_id", "opening_date", "retail_location_type"]], on="branch_id")
before = x[pd.to_datetime(x.order_date) < x.opening_date]
print("Orders before their branch opening date:", len(before), f"({len(before) / len(so):.1%})")
before.sales_channel.value_counts().to_frame("orders before opening")

Orders before their branch opening date: 38040 (7.0%)


,orders before opening
sales_channel,
Retail POS,27415
E-commerce,6783
Wholesale / reseller,3842


In [9]:
print(b[b.branch_name_ar.duplicated(keep=False)][["branch_id", "branch_name_ar", "governorate_ar", "address_ar"]], "\n")

for t, col in [("sales_orders", "branch_id"), ("employees", "branch_id"), ("inventory_transfers", "destination_branch_id")]:
    missing = (~tables[t][col].isin(b.branch_id)).sum()
    print(f"{t}.{col}: {tables[t][col].nunique()} branches used, {missing} codes not found in branches")

dd = tables["data_dictionary"]
print("\nColumns in dictionary but NOT in file:",
      set(dd[dd.table_name == "branches.csv"].field_name) - set(tables["branches"].columns))

   branch_id branch_name_ar governorate_ar                address_ar
41     BR042      سعد زغلول    الإسماعيلية        190 شارع سعد زغلول
56     BR057      سعد زغلول         الفيوم  ممر أبو عميرة، سعد زغلول 

sales_orders.branch_id: 80 branches used, 0 codes not found in branches
employees.branch_id: 80 branches used, 0 codes not found in branches
inventory_transfers.destination_branch_id: 80 branches used, 0 codes not found in branches

Columns in dictionary but NOT in file: {'source_url'}


In [10]:
# does a branch's place or type change how much it sells?
bs = b.set_index("branch_id")
bs["store_orders"] = so[so.sales_channel == "Retail POS"].groupby("branch_id").size()
bs["in_mall"] = bs.index.isin(mall.branch_id)
print("In-store orders per branch: min", bs.store_orders.min(), "| max", bs.store_orders.max())
print("Average by type:", bs.groupby("retail_location_type").store_orders.mean().round(0).to_dict())
print("Average in mall vs not:", bs.groupby("in_mall").store_orders.mean().round(0).to_dict(), "\n")
bs.sort_values("store_orders")[["branch_name_ar", "governorate_ar", "retail_location_type", "store_orders"]].iloc[[0, 1, -2, -1]]

In-store orders per branch: min 4699 | max 5011
Average by type: {'Point of sale / reseller': 4863.0, 'Retail branch': 4846.0}
Average in mall vs not: {False: 4855.0, True: 4824.0} 



,branch_name_ar,governorate_ar,retail_location_type,store_orders
branch_id,,,,
BR067,بني سويف الجديدة,بني سويف,Retail branch,4699
BR020,طلعت حرب,القاهرة,Retail branch,4723
BR003,القطامية داون تاون,القاهرة,Retail branch,4987
BR056,سقالة,البحر الأحمر,Point of sale / reseller,5011


In [11]:
# company branches vs reseller points: same mix of channels?
online = so[so.sales_channel == "E-commerce"].groupby("branch_id").size()
print("Branches with online orders:", online.size, "| per branch: min", online.min(), "| max", online.max())
pd.crosstab(x.retail_location_type, x.sales_channel, normalize="index").round(3)

Branches with online orders: 80 | per branch: min 1148 | max 1354


sales_channel,E-commerce,Retail POS,Wholesale / reseller
retail_location_type,,,
Point of sale / reseller,0.182,0.719,0.099
Retail branch,0.180,0.719,0.101


In [12]:
# where does a branch code appear in other tables?
for name, df in tables.items():
    cols = [col for col in df.columns if "branch" in col or col == "stock_location_id"]
    if cols and name != "branches":
        print(f"{name:20s}", cols)

employees            ['branch_id']
inventory_transfers  ['destination_branch_id']
monthly_stock        ['stock_location_id']
operating_expenses   ['branch_or_department_id']
sales_order_items    ['branch_id']
sales_orders         ['branch_id']


## Step 2: `customers`

In [13]:
c = tables["customers"]
print(c.shape)
c.head()

(85000, 6)


,customer_id,customer_segment,customer_name_ar,governorate_ar,joined_date,age_band
0,C0000001,Retail,كريم حمادة,دمياط,10/19/2018,55+
1,C0000002,Retail,مريم خليل,أسوان,10/19/2018,18-24
2,C0000003,Retail,خالد محمد,القاهرة,10/19/2018,45-54
3,C0000004,Retail,أحمد منصور,المنوفية,10/19/2018,35-44
4,C0000005,Retail,رانيا إبراهيم,المنوفية,10/19/2018,18-24


In [14]:
print("Empty cells:", c.isna().sum().sum())
print("Duplicate customer_id:", c.customer_id.duplicated().sum())
print("Columns in dictionary but NOT in file:",
      set(dd[dd.table_name == "customers.csv"].field_name) - set(c.columns))
print("Unique names:", c.customer_name_ar.nunique(), "for", len(c), "customers")
c.customer_name_ar.value_counts().head(3).to_frame("customers with this name")

Empty cells: 0
Duplicate customer_id: 0
Columns in dictionary but NOT in file: {'loyalty_tier'}
Unique names: 2233 for 85000 customers


,customers with this name
customer_name_ar,
يارا الشرقاوي,184
عبدالله علي,184
منى النجار,181


In [15]:
print(c.customer_segment.value_counts(), "\n")
print(c.age_band.value_counts().sort_index(), "\n")
gov = c.governorate_ar.value_counts()
print("Customers per governorate: min", gov.min(), "| max", gov.max(), "| governorates", len(gov), "\n")
c["joined_date"] = pd.to_datetime(c.joined_date, format="%m/%d/%Y")
c.joined_date.dt.year.value_counts().sort_index().to_frame("new customers")

customer_segment
Retail          83334
B2B reseller     1666
Name: count, dtype: int64 

age_band
18-24    16952
25-34    17058
35-44    16895
45-54    16945
55+      17150
Name: count, dtype: int64 

Customers per governorate: min 3956 | max 4134 | governorates 21 



,new customers
joined_date,
2018,2153
2019,10618
2020,10647
2021,10617
2022,10618
2023,10618
2024,10647
2025,10617
2026,8465


In [16]:
so = tables["sales_orders"]
print("Orders with NO customer_id:", so.customer_id.isna().sum(), f"({so.customer_id.isna().mean():.1%})")
print("customer_id in orders not found in customers:", (~so.customer_id.dropna().isin(c.customer_id)).sum())

buyers = so.dropna(subset=["customer_id"])
print("Customers who bought:", buyers.customer_id.nunique(),
      "| Registered but never bought:", (~c.customer_id.isin(buyers.customer_id)).sum())

first = pd.to_datetime(buyers.groupby("customer_id").order_date.min())
gap = (first - c.set_index("customer_id").joined_date.reindex(first.index)).dt.days
print("First order BEFORE joining:", (gap < 0).sum(), "| median days from joining to first order:", gap.median())
print("Orders per customer:", buyers.groupby("customer_id").size().describe().round(1).to_dict())

Orders with NO customer_id: 102961 (19.1%)
customer_id in orders not found in customers: 0
Customers who bought: 74660 | Registered but never bought: 10340
First order BEFORE joining: 0 | median days from joining to first order: 139.0
Orders per customer: {'count': 74660.0, 'mean': 5.9, 'std': 4.4, 'min': 1.0, '25%': 2.0, '50%': 5.0, '75%': 8.0, 'max': 43.0}


In [17]:
# is the missing customer only walk-in shoppers? check every channel
print(so.customer_id.isna().groupby(so.sales_channel).mean().round(3).to_frame("share without customer"), "\n")
online_unknown = so[(so.sales_channel == "E-commerce") & so.customer_id.isna()]
print("Online orders without a customer:", len(online_unknown),
      "| shipped to someone:", online_unknown.sales_order_id.isin(tables["delivery_shipments"].sales_order_id).sum())

                      share without customer
sales_channel                               
E-commerce                             0.189
Retail POS                             0.191
Wholesale / reseller                   0.193 

Online orders without a customer: 18430 | shipped to someone: 18430


In [18]:
seg = buyers.merge(c[["customer_id", "customer_segment"]], on="customer_id")
print(pd.crosstab(seg.sales_channel, seg.customer_segment), "\n")

br = tables["branches"][["branch_id", "governorate_ar"]]
m = buyers.merge(c[["customer_id", "governorate_ar"]], on="customer_id") \
          .merge(br, on="branch_id", suffixes=("_customer", "_branch"))
store = m[m.sales_channel == "Retail POS"]
print("In-store orders where customer lives in the SAME governorate as the branch:",
      f"{(store.governorate_ar_customer == store.governorate_ar_branch).mean():.1%}",
      "(random chance with 21 governorates = 4.8%)")

customer_segment      B2B reseller  Retail
sales_channel                             
E-commerce                    1503   77727
Retail POS                    6233  307885
Wholesale / reseller           851   42840 

In-store orders where customer lives in the SAME governorate as the branch: 4.8% (random chance with 21 governorates = 4.8%)


In [19]:
# how many different branches does one customer visit?
visits = buyers[buyers.sales_channel == "Retail POS"].merge(br, on="branch_id")
per_customer = visits.groupby("customer_id").agg(visits=("sales_order_id", "size"),
                                                 branches=("branch_id", "nunique"),
                                                 governorates=("governorate_ar", "nunique"))
regular = per_customer[per_customer.visits >= 5]
print("Customers with 5+ store visits:", len(regular))
regular.mean().round(1).to_frame("average per customer")

Customers with 5+ store visits: 27707


,average per customer
visits,7.7
branches,7.3
governorates,5.6


In [20]:
# where does a customer lead? customer_id -> sales_orders -> sales_order_id -> other tables
for name, df in tables.items():
    cols = [col for col in df.columns if col in ("customer_id", "sales_order_id")]
    if cols and name != "customers":
        print(f"{name:20s}", cols)

delivery_shipments   ['sales_order_id']
sales_order_items    ['sales_order_id']
sales_orders         ['sales_order_id', 'customer_id']
sales_payments       ['sales_order_id']
sales_returns        ['sales_order_id']


## Step 3: `products_catalog`

In [21]:
p = tables["products_catalog"]
print(p.shape)
p.head()

(139, 6)


,product_id,product_name_ar,category_en,audience,listed_price_EGP,source_url
0,PR0001,3 * شراب رجالي كلاسيك-لايكرا 1,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
1,PR0002,3 * شراب رجالي كلاسيك-لايكرا 2,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
2,PR0003,3 * شراب رجالي كلاسيك-لايكرا 3,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
3,PR0004,3 * شراب رجالي كلاسيك-لايكرا 4,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
4,PR0005,3 * شراب رجالي كلاسيك-لايكرا 5,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1


In [22]:
print("Empty cells:", p.isna().sum().sum(),
      "| duplicate ids:", p.product_id.duplicated().sum(),
      "| duplicate names:", p.product_name_ar.duplicated().sum())
p.category_en.value_counts().to_frame("products")

Empty cells: 0 | duplicate ids: 0 | duplicate names: 0


,products
category_en,
Socks,76
Men tops,9
Kids underwear,8
Women tops,7
Men boxers,7
Women bras,6
Men underwear,5
Home textiles,4
Boys pajamas,4


In [23]:
# who is each category for? (look at Women bras -> Unisex)
pd.crosstab(p.category_en, p.audience)

audience,Boys,Girls,Kids,Men,Unisex,Women
category_en,,,,,,
Accessories,0,0,0,0,1,0
Boys pajamas,4,0,0,0,0,0
Girls pajamas,0,4,0,0,0,0
Girls underwear,0,0,0,0,1,0
Home textiles,0,0,0,0,4,0
Kids underwear,0,0,8,0,0,0
Men bottoms,0,0,0,3,0,0
Men boxers,0,0,0,2,5,0
Men tops,0,0,0,7,2,0


In [24]:
# price per category (price is for the whole pack, 2026 website price)
p.groupby("category_en").listed_price_EGP.agg(["count", "min", "median", "max"]).sort_values("median")

,count,min,median,max
category_en,,,,
Kids underwear,8,190,190.0,190
Women tops,7,190,190.0,550
Women bottoms,1,255,255.0,255
Socks,76,220,290.0,485
Men bottoms,3,310,310.0,395
Accessories,1,435,435.0,435
Girls pajamas,4,500,500.0,500
Boys pajamas,4,500,500.0,500
Women bras,6,225,505.0,505


In [25]:
# near-duplicate products: same name with a number at the end
base = p.product_name_ar.str.replace(r"[\d\-]+\s*$", "", regex=True).str.strip()
base.value_counts().head(8).to_frame("copies")

,copies
product_name_ar,
3 * شراب رجالي كلاسيك-لايكرا,57
بكيني كاندي مطبوع 2 قطعة توينز,8
ايزي توب قطن,4
بكيني رجالي مارسريزية,4
بيجامة صيفي اولادي - كود,4
بيجامة صيفي بناتي - كود,4
بيجامة شتوي حريمي كود,2
3 طقم بناتى هوت شورت,1


## Step 4: `sku_variants`

In [26]:
v = tables["sku_variants"]
print(v.shape)
v.head(10)

(695, 5)


,sku_id,product_id,size,color_ar,price_multiplier
0,SKU000001,PR0001,Free,أبيض,0.950
1,SKU000002,PR0001,M,أسود,0.975
2,SKU000003,PR0001,L,كحلي,1.000
3,SKU000004,PR0001,XL,رمادي,1.025
4,SKU000005,PR0001,Free,بيج,1.050
5,SKU000006,PR0002,M,رمادي,0.950
6,SKU000007,PR0002,L,بيج,0.975
7,SKU000008,PR0002,XL,أزرق,1.000
8,SKU000009,PR0002,Free,نبيتي,1.025
9,SKU000010,PR0002,M,وردي,1.050


In [27]:
print("Empty cells:", v.isna().sum().sum(), "| duplicate sku_id:", v.sku_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "sku_variants.csv"].field_name) - set(v.columns))
print("SKUs per product:", v.groupby("product_id").size().value_counts().to_dict(),
      "| products with no SKU:", (~p.product_id.isin(v.product_id)).sum())
print("Sizes:", v["size"].value_counts().to_dict())
print("Colours:", v.color_ar.value_counts().to_dict())
v.price_multiplier.value_counts().sort_index().to_frame("SKUs")

Empty cells: 0 | duplicate sku_id: 0
Columns in dictionary but NOT in file: set()
SKUs per product: {5: 139} | products with no SKU: 0
Sizes: {'XL': 142, 'L': 141, 'M': 136, 'Free': 113, '2XL': 28, '4XL': 28, '3XL': 27, '3': 14, '6': 14, '2': 13, '4': 13, '5': 13, '7': 13}
Colours: {'أبيض': 87, 'أسود': 87, 'كحلي': 87, 'رمادي': 87, 'بيج': 87, 'نبيتي': 87, 'وردي': 87, 'أزرق': 86}


,SKUs
price_multiplier,
0.950,139
0.975,139
1.000,139
1.025,139
1.050,139


In [28]:
# 5 SKUs per product, each one a different colour -> every colour exists in ONE size only
print("Products with the same colour twice:", v.groupby("product_id").color_ar.apply(lambda s: s.duplicated().any()).sum())
print("Products with the same size twice:", v.groupby("product_id")["size"].apply(lambda s: s.duplicated().any()).sum())
v[v.product_id.isin(["PR0001", "PR0060"])]

Products with the same colour twice: 0
Products with the same size twice: 89


,sku_id,product_id,size,color_ar,price_multiplier
0,SKU000001,PR0001,Free,أبيض,0.950
1,SKU000002,PR0001,M,أسود,0.975
2,SKU000003,PR0001,L,كحلي,1.000
3,SKU000004,PR0001,XL,رمادي,1.025
4,SKU000005,PR0001,Free,بيج,1.050
295,SKU000296,PR0060,4XL,أسود,0.950
296,SKU000297,PR0060,M,كحلي,0.975
297,SKU000298,PR0060,L,رمادي,1.000
298,SKU000299,PR0060,XL,بيج,1.025
299,SKU000300,PR0060,2XL,أزرق,1.050


In [29]:
# price_multiplier follows the SKU's position inside its product (1st, 2nd, ...), not its size
v["position"] = v.groupby("product_id").cumcount() + 1
pd.crosstab(v.position, v.price_multiplier)

price_multiplier,0.95,0.975,1.0,1.025,1.05
position,,,,,
1,139,0,0,0,0
2,0,139,0,0,0
3,0,0,139,0,0
4,0,0,0,139,0
5,0,0,0,0,139


In [30]:
# odd sizes: a girls' product and towels
vp = v.merge(p[["product_id", "product_name_ar", "category_en"]], on="product_id")
vp[vp.category_en.isin(["Girls underwear", "Home textiles"])].groupby(["product_id", "product_name_ar"])["size"].apply(" ".join).to_frame("sizes")

,,sizes
product_id,product_name_ar,
PR0058,3 طقم بناتى هوت شورت,2XL 3XL 4XL M L
PR0062,3 قطع فوط مقاس 100*50,L XL Free L XL
PR0102,بشكير احرام قطونيل,XL Free L XL Free
PR0103,بشكير قطونيل كبير ( 90x160 ),Free L XL Free L
PR0104,بشكير قطونيل وسط ( 70x140 ),L XL Free L XL


In [31]:
# products whose NAME says one colour, but the SKUs have 5 colours
colour_word = r"(?:^|[\s\-])(?:ابيض|أبيض|اسود|أسود|بيج|ازرق|أزرق|كحلي|رمادي|نبيتي|وردي)(?:$|[\s\-])"
named = p[p.product_name_ar.str.contains(colour_word, regex=True) & ~p.product_name_ar.str.contains("الوان")]
print("Products with one colour in their name:", len(named))
vp[vp.product_id.isin(named.product_id)].groupby("product_name_ar").color_ar.apply(" - ".join).to_frame("SKU colours")

Products with one colour in their name: 14


,SKU colours
product_name_ar,
3* شراب ميني سوكيت أبيض,أزرق - نبيتي - وردي - أبيض - أسود
3*شراب قدم أبيض,أبيض - أسود - كحلي - رمادي - بيج
3*شراب قدم أزرق فاتح,أسود - كحلي - رمادي - بيج - أزرق
3*شراب قدم ازرق غامق,وردي - أبيض - أسود - كحلي - رمادي
3*شراب قدم اسود,كحلي - رمادي - بيج - أزرق - نبيتي
3*شراب قدم بيج,أبيض - أسود - كحلي - رمادي - بيج
3*شراب ميني سوكيت أزرق غامق,وردي - أبيض - أسود - كحلي - رمادي
3*شراب ميني سوكيت أسود,كحلي - رمادي - بيج - أزرق - نبيتي
برا بدن دانتیل بوش اب-ابيض,أزرق - نبيتي - وردي - أبيض - أسود


In [32]:
# is the price_multiplier really used in the selling price? compare each SKU with its product in the same month
si = tables["sales_order_items"]
s = si.merge(v[["sku_id", "price_multiplier"]], on="sku_id")
month = s.order_date.str[:7]
s["price_vs_product"] = s.unit_price_EGP / s.groupby([s.product_id, month]).unit_price_EGP.transform("mean")
v["units"] = v.sku_id.map(si.groupby("sku_id").quantity.sum())
pd.DataFrame({"selling price vs product": s.groupby("price_multiplier").price_vs_product.mean().round(3),
              "units sold per SKU": v.groupby("price_multiplier").units.mean().round(0)})

,selling price vs product,units sold per SKU
price_multiplier,,
0.950,0.951,4896.0
0.975,0.975,4889.0
1.000,1.000,4898.0
1.025,1.025,4885.0
1.050,1.049,4883.0


In [33]:
# inside the same product, do size or colour change how much sells?
v["vs_product"] = v.units / v.groupby("product_id").units.transform("mean")
print("Units vs own product, by size:", v.groupby("size").vs_product.mean().round(2).to_dict())
print("Units vs own product, by colour:", v.groupby("color_ar").vs_product.mean().round(2).to_dict(), "\n")
clothes = v[v.product_id.isin(v[v["size"].isin(["2XL", "3XL", "4XL"])].product_id)]
print("Clothes with sizes M to 4XL:", clothes.product_id.nunique(), "products")
clothes.groupby("size").units.mean().round(0).to_frame("units sold per SKU")

Units vs own product, by size: {'2': 1.0, '2XL': 1.01, '3': 1.02, '3XL': 1.0, '4': 1.0, '4XL': 1.0, '5': 0.98, '6': 1.01, '7': 0.98, 'Free': 1.0, 'L': 1.0, 'M': 0.99, 'XL': 1.0}
Units vs own product, by colour: {'أبيض': 0.99, 'أزرق': 0.99, 'أسود': 1.0, 'بيج': 1.0, 'رمادي': 1.0, 'كحلي': 1.02, 'نبيتي': 1.0, 'وردي': 1.0} 

Clothes with sizes M to 4XL: 34 products


,units sold per SKU
size,
2XL,8942.0
3XL,8998.0
4XL,9009.0
L,9115.0
M,8783.0
XL,9080.0


In [34]:
# where does sku_id appear?
for name, df in tables.items():
    if "sku_id" in df.columns and name != "sku_variants":
        print(f"{name:26s}", df.sku_id.nunique(), "SKUs used |", (~df.sku_id.isin(v.sku_id)).sum(), "unknown codes")
print("\nmonthly_stock is kept by:", [col for col in tables["monthly_stock"].columns if "product" in col or "sku" in col])

inventory_transfer_items   695 SKUs used | 0 unknown codes
purchase_order_items       695 SKUs used | 0 unknown codes
sales_order_items          695 SKUs used | 0 unknown codes
sales_returns              695 SKUs used | 0 unknown codes

monthly_stock is kept by: ['product_category']
